# Week 2 · Day 5 — Capstone: Construction Operations Analytics

**Business First Prompt:**  
You've been brought in as a data analyst for a mid-size Canadian general contractor. The ops team needs a full performance review: which projects are over budget, which subcontractors are generating the most value, which site managers are running the tightest ships, and where inspection failures are clustering. 

**KPI Framework:**  
- Project cost overrun rate (actual_cost vs contract_value)
- Subcontractor revenue and utilization by trade
- Site manager project load and completion rate
- Inspection pass/fail rate by project type and province
- Work order cycle time (issued → completion)

**What you'll demonstrate today:**
1. Multi-table JOINs across a real operations schema
2. GROUP BY aggregations with business-relevant KPIs
3. Subqueries and CTEs for layered analysis
4. Window functions for ranking and deviation
5. CASE WHEN for business classification
6. Full capstone pipeline combining all of the above

**Schema:**
- `projects` — project_id, client_id, manager_id, type, city, province, start_date, end_date, status, contract_value, actual_cost
- `clients` — client_id, company, city, province, segment
- `site_managers` — manager_id, name, province, hire_date, status
- `subcontractors` — sub_id, company, trade, province, hourly_rate
- `work_orders` — order_id, project_id, sub_id, issued_date, completion_date, status, labour_hours, labour_cost, materials_cost
- `inspections` — inspection_id, project_id, inspection_date, type, result, inspector_name

---

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('construction_w5.db')
print('Connected to construction_w5.db')

Connected to construction_w5.db


In [42]:
for table in ['projects', 'clients', 'site_managers', 'subcontractors', 'work_orders', 'inspections']:
    print(f'\n--- {table} ---')
    df = pd.read_sql_query(f'SELECT * FROM {table} LIMIT 3', conn)
    print(f'Columns: {list(df.columns)}')
    display(df)


--- projects ---
Columns: ['project_id', 'client_id', 'manager_id', 'type', 'city', 'province', 'start_date', 'end_date', 'status', 'contract_value', 'actual_cost']


,project_id,client_id,manager_id,type,city,province,start_date,end_date,status,contract_value,actual_cost
0,1,2,1,Infrastructure,Vancouver,BC,2022-09-08,2023-03-31,In Progress,3748238.27,NaN
1,2,9,2,Infrastructure,Calgary,AB,2022-02-02,2022-05-18,Completed,1288530.38,1355717.08
2,3,1,4,Infrastructure,Montreal,QC,2023-07-13,2024-05-12,In Progress,2383742.97,NaN



--- clients ---
Columns: ['client_id', 'company', 'city', 'province', 'segment']


,client_id,company,city,province,segment
0,1,Maple Developments Inc,Toronto,ON,Developer
1,2,Pacific Build Corp,Vancouver,BC,Developer
2,3,Prairie Infrastructure Ltd,Calgary,AB,Government



--- site_managers ---
Columns: ['manager_id', 'name', 'province', 'hire_date', 'status']


...

,sub_id,company,trade,province,hourly_rate
0,1,Volt Pro Electric,Electrical,ON,95.0
1,2,Pacific Frame & Truss,Framing,BC,78.0
2,3,Solid Ground Concrete,Concrete,AB,85.0



--- work_orders ---
Columns: ['order_id', 'project_id', 'sub_id', 'issued_date', 'completion_date', 'status', 'labour_hours', 'labour_cost', 'materials_cost']


,order_id,project_id,sub_id,issued_date,completion_date,status,labour_hours,labour_cost,materials_cost
0,1,1,12,2022-11-04,None,Pending,182.3,18047.7,35629.47
1,2,1,7,2022-10-22,None,Pending,98.6,7099.2,36533.48
2,3,1,7,2022-10-21,None,Pending,301.8,21729.6,35455.64



--- inspections ---
Columns: ['inspection_id', 'project_id', 'inspection_date', 'type', 'result', 'inspector_name']


,inspection_id,project_id,inspection_date,type,result,inspector_name
0,1,1,2022-12-23,Fire Safety,Pass,R. Thompson
1,2,1,2022-11-17,Fire Safety,Fail,C. Osei
2,3,1,2023-02-03,Electrical,Fail,D. Patel


---
## Section 1 — Project Cost Analysis

The finance team wants to know which projects went over budget and by how much. `actual_cost` is only populated for Completed projects — all others are NULL.

**Key columns:**
- `contract_value` — what the client agreed to pay
- `actual_cost` — what it actually cost to deliver
- Overrun = `actual_cost - contract_value`
- Overrun % = `ROUND(100.0 * (actual_cost - contract_value) / contract_value, 1)`

---

In [43]:
# q1 — Completed projects: contract value, actual cost, overrun amount, overrun %
q1 = """
SELECT   project_id,
        contract_value,
        actual_cost,
        (actual_cost - contract_value) AS overrun_amount,
        ROUND((actual_cost - contract_value) * 100 / contract_value, 1) AS 'overrun %'
FROM projects 
WHERE status = 'Completed' AND actual_cost > contract_value
ORDER BY overrun_amount DESC 
"""
df1 = pd.read_sql_query(q1, conn)
display(df1)

,project_id,contract_value,actual_cost,overrun_amount,overrun %
0,28,4852732.22,5922986.75,1070254.53,22.1
1,18,4663218.43,5602804.49,939586.06,20.1
2,19,3726561.08,4383961.12,657400.04,17.6
3,41,4214326.34,4564131.55,349805.21,8.3
4,23,1337473.47,1621242.55,283769.08,21.2
5,33,2241318.35,2428398.76,187080.41,8.3
6,14,2077644.73,2261215.80,183571.07,8.8
7,26,537910.20,613909.08,75998.88,14.1
8,2,1288530.38,1355717.08,67186.70,5.2
9,7,2055599.57,2120071.64,64472.07,3.1


In [44]:
# q2 — Average overrun % by project type (Residential / Commercial / Infrastructure)

q2 = """
SELECT type,
        ROUND(AVG((actual_cost - contract_value) * 100 / contract_value), 2) AS avg_overrun_pct
FROM projects 
WHERE status = 'Completed' AND actual_cost > contract_value
GROUP BY type
ORDER BY avg_overrun_pct DESC
"""
df2 = pd.read_sql_query(q2, conn)
display(df2)

,type,avg_overrun_pct
0,Commercial,13.11
1,Infrastructure,9.54
2,Residential,9.39


In [45]:
# q3 — Flag projects as OVER / UNDER / ON BUDGET using CASE WHEN
q3 = """ SELECT project_id,
                type,
                (CASE WHEN actual_cost - contract_value = 0 THEN 'On Budget'
                WHEN actual_cost > contract_value THEN 'Over' ELSE 'Under' END) AS flag
                FROM projects
                WHERE status = 'Completed'
"""
df3 = pd.read_sql_query(q3, conn)
display(df3)

,project_id,type,flag
0,2,Infrastructure,Over
1,7,Commercial,Over
2,10,Infrastructure,Over
3,14,Infrastructure,Over
4,18,Infrastructure,Over
5,19,Residential,Over
6,23,Infrastructure,Over
7,25,Residential,Over
8,26,Commercial,Over
9,28,Commercial,Over


---
## Section 2 — Subcontractor Performance

The ops team wants to know which subcontractors are generating the most revenue and how their cost structure breaks down between labour and materials.

**Key columns in work_orders:**
- `labour_cost` — hours × rate for that work order
- `materials_cost` — materials billed on that work order
- Total cost = `labour_cost + materials_cost`

---

In [46]:
# q4 — Subcontractor total revenue and work order count

q4 = """
SELECT  s.sub_id,
        (s.hourly_rate * wo.labour_hours) AS revenue,
        COUNT(wo.order_id) AS order_count
FROM subcontractors s JOIN work_orders wo ON s.sub_id = wo.sub_id
GROUP BY s.sub_id
ORDER BY revenue DESC
"""
df4 = pd.read_sql_query(q4, conn)
display(df4)

,sub_id,revenue,order_count
0,10,46422.6,22
1,5,42147.0,29
2,3,41871.0,22
3,8,36960.0,25
4,6,30870.4,25
5,1,21470.0,18
6,12,18047.7,15
7,9,15299.6,17
8,2,14258.4,26
9,11,10011.6,23


In [47]:
# q5 — Top subcontractor per trade by total revenue (window function)
q5 = """
       WITH ranked AS ( SELECT s.sub_id,
                s.company,
                s.trade,
                SUM(s.hourly_rate * wo.labour_hours) AS total_revenue,
                RANK() OVER (PARTITION BY s.trade ORDER BY SUM(s.hourly_rate * wo.labour_hours)) AS rnk
                FROM subcontractors s JOIN work_orders wo ON s.sub_id = wo.sub_id
                GROUP BY s.sub_id, s.company, s.trade)
                
    
SELECT sub_id,
        company,
        trade,
        total_revenue
        FROM ranked
        WHERE rnk = 1
        ORDER BY total_revenue DESC


"""
df5 = pd.read_sql_query(q5, conn)
display(df5)

,sub_id,company,trade,total_revenue
0,10,Rockies HVAC Inc,HVAC,564156.6
1,8,Maritime Concrete Works,Concrete,528904.0
2,11,Metro Frame Builders,Framing,437108.4
3,9,Capital Plumbing Group,Plumbing,388304.4
4,12,WestCoast Electrical,Electrical,372566.7


In [48]:
# q6 — Average work order cycle time by subcontractor trade
q6 = """    SELECT  s.trade,
            ROUND(AVG(JULIANDAY(completion_date) - JULIANDAY(issued_date)) , 2) AS avg_order_cycle
            FROM subcontractors s JOIN work_orders wo ON s.sub_id = wo.sub_id
            WHERE completion_date IS NOT NULL
            GROUP BY s.trade
            ORDER BY avg_order_cycle DESC
"""
df6 = pd.read_sql_query(q6, conn)
display(df6)

,trade,avg_order_cycle
0,Electrical,28.71
1,Concrete,28.17
2,Plumbing,27.45
3,Framing,26.32
4,HVAC,25.11


---
## Section 3 — Site Manager Scorecard

HR wants a performance profile for each site manager: how many projects they're running, what's their completion rate, and how does their average contract value compare to the company average.

---

In [49]:
# q7 — Site manager project load and completion rate
q7 = """
   WITH site_managers_table AS  ( SELECT  p.manager_id,   
                                 sm.name,
                                 COUNT(p.project_id) AS total_projects,
                                (SELECT  COUNT(project_id) AS project_count
                                 FROM projects
                                WHERE status = 'In Progress' AND manager_id = p.manager_id) AS current_projects
                                FROM projects p JOIN site_managers sm ON p.manager_id = sm.manager_id
                                GROUP BY p.manager_id, sm.name),


        completed AS ( SELECT manager_id,
                        COUNT(project_id) AS completed_count
                        FROM projects
                        WHERE status = 'Completed'
                        GROUP BY manager_id)
        
        SELECT  sm.name,
                sm.manager_id,
                total_projects,
                ROUND( completed_count * 100.0 /total_projects, 2) AS completion_rate
                FROM site_managers_table sm JOIN completed c ON sm.manager_id = c.manager_id
        ORDER BY completion_rate DESC
"""
df7 = pd.read_sql_query(q7, conn)
display(df7)

,name,manager_id,total_projects,completion_rate
0,Carlos Rivera,8,3,66.67
1,Jean Tremblay,1,7,57.14
2,David Park,4,10,50.00
3,Christine Bouchard,5,4,50.00
4,Priya Mehta,7,5,40.00
5,Mike Okafor,2,9,33.33
6,Sara Singh,3,10,10.00


In [50]:
# q8 — Manager avg contract value vs company benchmark (scalar CTE)
q8 = """

WITH per_manager AS (   SELECT manager_id,
                                SUM(contract_value) AS total_contract
                        FROM projects
                        GROUP BY manager_id),

    avg AS( SELECT ROUND(AVG(total_contract), 2) AS company_avg
            FROM per_manager)

    SELECT  manager_id,
            total_contract,
            company_avg
    FROM per_manager CROSS JOIN avg
    ORDER BY total_contract DESC

"""
df8 = pd.read_sql_query(q8, conn)
display(df8)

,manager_id,total_contract,company_avg
0,2,25882045.25,16616462.96
1,3,25688030.24,16616462.96
2,4,25557053.93,16616462.96
3,1,24449013.90,16616462.96
4,5,13628956.06,16616462.96
5,7,11591375.32,16616462.96
6,6,4272696.17,16616462.96
7,8,1862532.81,16616462.96


---
## Section 4 — Inspection Analysis

The compliance team needs to know where failures are clustering — by province, project type, and inspection type.

---

In [51]:
# q9 — Inspection pass/fail rate by province
q9 = """
      WITH province_stats AS (SELECT p.province,
                               SUM(CASE WHEN i.result = 'Pass' THEN 1 ELSE 0 END) AS pass_count,
                                SUM(CASE WHEN i.result = 'Fail' THEN 1 ELSE 0 END) AS fail_count,
                                COUNT(i.inspection_id) AS total_inspections  
                                FROM projects p 
                                 JOIN inspections i ON p.project_id = i.project_id
                                GROUP BY p.province)
                                
  

SELECT province,
       pass_count,
       fail_count,
       total_inspections,
       ROUND(pass_count * 100.0 / (pass_count + fail_count), 2) AS pass_rate,
       ROUND(fail_count * 100.0 / (pass_count + fail_count), 2) AS fail_rate
FROM province_stats
ORDER BY pass_rate DESC
        
"""
df9 = pd.read_sql_query(q9, conn)
display(df9)

,province,pass_count,fail_count,total_inspections,pass_rate,fail_rate
0,ON,19,18,60,51.35,48.65
1,BC,8,10,22,44.44,55.56
2,NS,1,2,5,33.33,66.67
3,QC,4,8,18,33.33,66.67
4,AB,6,14,32,30.00,70.00


In [52]:
# q10 — Most recent inspection result per project (ROW_NUMBER deduplication)
q10 = """

          WITH  date_rank AS (  SELECT project_id,
                            inspection_id,
                            inspection_date,
                            result,
                            ROW_NUMBER() OVER (PARTITION BY project_id ORDER BY inspection_date DESC) AS rnk
                            FROM inspections)

    SELECT project_id,
            inspection_date,
            result
    FROM date_rank
    WHERE rnk=1

        
"""
df10 = pd.read_sql_query(q10, conn)
display(df10)

,project_id,inspection_date,result
0,1,2023-02-03,Fail
1,2,2022-07-25,Fail
2,3,2023-11-18,Fail
3,4,2023-04-07,Pending
4,5,2023-06-09,Fail
5,6,2023-12-07,Fail
6,7,2023-03-04,Pass
7,8,2022-12-11,Fail
8,9,2023-01-18,Pass
9,10,2022-08-08,Pass


---
## ★ CAPSTONE — Full Project Performance Report

**Business ask:**  
The executive team wants one report that scores every completed project across four dimensions:
- Budget performance (overrun vs contract)
- Work order efficiency (total work order cost vs actual project cost)
- Inspection health (pass rate for that project)
- Overall project rank within its province by contract value

**Requirements:**
1. Minimum 4 CTEs
2. At least one scalar CTE + CROSS JOIN (fleet-level benchmark)
3. CASE WHEN budget classification: OVER / ON / UNDER BUDGET



In [53]:
# ★ CAPSTONE — Full Project Performance Report\
#every completed project 
#Budget performance (overrun vs contract)
#Work order efficiency (total work order cost vs actual project cost)
#Inspection health (pass rate for that project)
# Overall project rank within its province by contract value


q_capstone = """

WITH c_projects AS (  SELECT project_id,
                        type,
                        city,
                        province,
                        start_date,
                        end_date,
                        contract_value,
                        actual_cost,
                        CASE WHEN actual_cost > contract_value THEN 'Over Budget'
                              WHEN contract_value = actual_cost THEN 'On Budget'
                              ELSE 'Under Budget' END AS budget_performance,
                        RANK() OVER (PARTITION BY province ORDER BY contract_value DESC) AS rnk,
                        JULIANDAY(end_date) - JULIANDAY(start_date) AS project_time_total,
                        ( contract_value- actual_cost) AS Profit_Loss
                        FROM projects
                        WHERE status = 'Completed'),

avg_time AS (   SELECT ROUND(AVG(project_time_total), 2) AS project_avg_time
                        FROM c_projects),


project_expense AS ( SELECT  
                            project_id,
                            SUM(labour_cost + materials_cost) AS cost_per_project
                            FROM work_orders
                            GROUP BY project_id),
         
 inspection_health AS ( SELECT      project_id,
                                    COUNT(inspection_id) AS inspection_count,
                                    SUM(CASE WHEN result = 'Pass' THEN 1 ELSE 0 END) AS pass_count
                                    FROM inspections
                                    GROUP BY project_id),

inspection_pass_avg AS (SELECT ROUND(SUM(pass_count) * 100.0 / SUM(inspection_count), 2) AS pass_avg
                        FROM inspection_health)

                   
SELECT cp.project_id AS 'Project Id',
        cp.type AS 'Project Type',
        cp.city AS 'City',
        cp.province AS 'Province',
        cp.start_date AS 'Start Date',
         cp.end_date AS 'End Date',
         cp.project_time_total AS 'Project Time Total',
         project_avg_time AS 'AVG Project Time',
         cp.contract_value AS 'Contract Value',
        cp.actual_cost AS 'Actual Cost',
        cp.profit_loss AS 'Profit/Loss',
        cp.budget_performance AS 'Budget Performance',
        pe.cost_per_project AS 'Work Order Costs',
        ROUND(( pe.cost_per_project * 100.0 / cp.actual_cost ), 2) AS 'Work order pctg',
        cp.rnk AS 'Province Rank',
        ih.inspection_count AS 'Inspections Count',
        ih.pass_count AS 'Pass Count',
        pass_avg AS 'AVG Pass Inspections'
FROM c_projects cp JOIN project_expense pe ON cp.project_id = pe.project_id JOIN inspection_health ih ON cp.project_id = ih.project_id 
CROSS JOIN avg_time CROSS JOIN inspection_pass_avg 
ORDER BY rnk
"""
df_capstone = pd.read_sql_query(q_capstone, conn)
display(df_capstone)

,Project Id,Project Type,City,Province,Start Date,End Date,Project Time Total,AVG Project Time,Contract Value,Actual Cost,Profit/Loss,Budget Performance,Work Order Costs,Work order pctg,Province Rank,Inspections Count,Pass Count,AVG Pass Inspections
0,18,Infrastructure,Vancouver,BC,2023-05-16,2023-08-23,99.0,278.37,4663218.43,5602804.49,-939586.06,Over Budget,354863.19,6.33,1,3,3,27.74
1,28,Commercial,Edmonton,AB,2022-04-11,2022-08-04,115.0,278.37,4852732.22,5922986.75,-1070254.53,Over Budget,339979.69,5.74,1,1,0,27.74
2,42,Residential,Toronto,ON,2022-11-12,2023-08-15,276.0,278.37,4519892.02,4221717.52,298174.50,Under Budget,339375.96,8.04,1,3,0,27.74
3,45,Residential,Montreal,QC,2022-04-21,2023-01-31,285.0,278.37,4328317.94,4064175.79,264142.15,Under Budget,636851.69,15.67,1,1,1,27.74
4,49,Commercial,Halifax,NS,2022-07-14,2023-02-19,220.0,278.37,3616730.10,3076667.34,540062.76,Under Budget,413475.12,13.44,1,3,1,27.74
5,10,Infrastructure,Vancouver,BC,2022-02-27,2022-09-22,207.0,278.37,4073967.55,4116605.35,-42637.80,Over Budget,429958.57,10.44,2,4,1,27.74
6,26,Commercial,Montreal,QC,2022-02-25,2023-05-05,434.0,278.37,537910.20,613909.08,-75998.88,Over Budget,262597.07,42.77,2,2,0,27.74
7,33,Infrastructure,Halifax,NS,2022-02-10,2023-03-24,407.0,278.37,2241318.35,2428398.76,-187080.41,Over Budget,291969.84,12.02,2,2,0,27.74
8,37,Infrastructure,Edmonton,AB,2022-11-03,2024-01-06,429.0,278.37,4709917.52,4256429.92,453487.60,Under Budget,348755.26,8.19,2,4,0,27.74
9,41,Residential,Toronto,ON,2022-06-02,2023-06-06,369.0,278.37,4214326.34,4564131.55,-349805.21,Over Budget,194424.10,4.26,2,3,2,27.74
